# DishNet — Full Training Run
**Food-101 image classifier | 101 classes | trained from scratch**

Before running: make sure GPU is enabled.
Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2 — Clone repo and install dependencies
!git clone https://github.com/ali-002-code/food-classifier.git
%cd food-classifier
!pip install -r requirements.txt -q

In [ ]:
# Cell 3 — Verify model architecture
import sys
sys.path.insert(0, '.')
from src.model import DishNet
import torch

model = DishNet(num_classes=101)
dummy = torch.randn(4, 3, 128, 128)
out = model(dummy)
print(f"Input:      {tuple(dummy.shape)}")
print(f"Output:     {tuple(out.shape)}")
print(f"Parameters: {model.count_parameters():,}")
print("Architecture check passed.")

In [ ]:
# Cell 4 — Full training run (downloads Food-101 ~5GB, then trains 30 epochs)
# Expected time: ~3 hours on T4 GPU
# Expected result: ~55% top-1, ~82% top-5
!python main.py

In [ ]:
# Cell 5 — Display learning curves
from IPython.display import Image, display
display(Image('results/learning_curves.png'))

In [ ]:
# Cell 6 — Display confusion matrix
from IPython.display import Image, display
display(Image('results/confusion_matrix.png'))

In [ ]:
# Cell 7 — Print best and worst classified classes
import pandas as pd
df = pd.read_csv('results/per_class_metrics.csv', index_col=0)
print("Top 10 best classified dishes:")
print(df.head(10)[['precision','recall','f1-score','support']].to_string())
print("\nTop 10 worst classified dishes:")
print(df.tail(10)[['precision','recall','f1-score','support']].to_string())

In [ ]:
# Cell 8 — Download results to your machine
# Downloads: best model checkpoint, plots, and per-class metrics
from google.colab import files
import os

files_to_download = [
    'results/checkpoints/best_model.pt',
    'results/learning_curves.png',
    'results/confusion_matrix.png',
    'results/per_class_metrics.csv',
]

for f in files_to_download:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
    else:
        print(f"Missing: {f}")

In [ ]:
# Cell 9 — (Optional) Push results back to GitHub
# Only the plots and CSV — checkpoint is too large for GitHub
import subprocess

GITHUB_TOKEN = ""  # paste your GitHub PAT here (repo scope)
REPO = "ali-002-code/food-classifier"

if GITHUB_TOKEN:
    !git config user.email "alieddama@icloud.com"
    !git config user.name "Ali Eddama"
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/{REPO}.git
    !git add results/learning_curves.png results/confusion_matrix.png results/per_class_metrics.csv
    !git commit -m "Add training results: learning curves, confusion matrix, per-class metrics"
    !git push origin main
    print("Results pushed to GitHub.")
else:
    print("Skipped — paste your GitHub PAT into GITHUB_TOKEN to push results.")